In [ ]:
import pandas as pd
import numpy as np
import re
from datetime import datetime, timedelta
from pathlib import Path
import logging
from io import BytesIO
from apicall_input import main_api_call 

In [ ]:
def create_emptydf(start_date,end_date):
    """
    Creates empty DataFrame with date range
    Args:
        start_date (str): Start date in 'yyyy-mm-dd' format
        end_date (str): End date in 'yyyy-mm-dd' format
        
    Returns:
        empty (df): Eempty df ready for population
    """
    # Convert start_date to a consistent format
    if isinstance(start_date, str):
        start_date = start_date.split(' ')[0]  # Remove time if present
        start = datetime.strptime(start_date, '%Y-%m-%d')
    elif isinstance(start_date, datetime):
        start = start_date.date()  # Extract date part if datetime object
    else:
        raise ValueError("Invalid start_date format")

    # Convert end_date to a consistent format
    if isinstance(end_date, str):
        end_date = end_date.split(' ')[0]  # Remove time if present
        end = datetime.strptime(end_date, '%Y-%m-%d')
    elif isinstance(end_date, datetime):
        end = end_date.date()  # Extract date part if datetime object
    else:
        raise ValueError("Invalid end_date format")


    date_range = pd.date_range(start, end)

    df = pd.DataFrame({'Date': date_range})
    
    df['Date'] = df['Date'].dt.strftime('%Y-%m-%d')
    df['nr. sessions'] = 0
    df['total km'] = 0.00
    df['km Z3-4'] = 0.00
    df['km Z5-T1-T2'] = 0.00
    df['hours alternative'] = 0.00
    return df

In [ ]:
def convert_to_day_approach(df):
    """
    Converts the DataFrame to a day approach format.
    
    Args:
        df (DataFrame): The DataFrame to convert.
        
    Returns:
        DataFrame: The converted DataFrame into a format with 7 lagging days 
        before each date in the format 

    """
    feature_cols = ['nr. sessions', 'total km', 'km Z3-4', 'km Z5-T1-T2', 'hours alternative']
    df_converted = pd.DataFrame()
    for i in range(0,7):
        for col in feature_cols:
            df_converted[f'{col}.{i}'] = df[col].shift(i)  
    df_converted['Date'] = df['Date']
    # drop rows with NaN values using dropna() with index as the row
    df_converted = df_converted.dropna()

    # replace the name of the column with the name of the column without the last 2 characters
    df_converted = df_converted.rename(columns={col: col[:-2] for col in df_converted.columns if col.endswith('.0')})


    # return df_lagged
    return df_converted 

In [ ]:
def read_df_memory(saved_activity_dictionary):
    '''
    separates out two lists of dictionaries from the downloaded_activities list

    Args: 
        df_memory (list): the list of dictionaries that contains all the downloaded activities in the format:
        {'filename': filename, 'data': data}

    Returns:
        run_activities (list), other_activities (list): two lists of dictionaries that contain the activities that are running and the rest of the activities
    '''
    run_activities = []
    other_activities = []
    for item in saved_activity_dictionary:
        filename = item["filename"]
        data = item["df"]
        # if filename  matches '*Running_*.csv':
        if re.match(r".*running_.*\.csv$", filename.lower()):
            run_activities.append({'filename': filename, 'df': data})
        else:
            other_activities.append({'filename': filename, 'df': data})
    for item in run_activities:
        filename = item["filename"]
        print(filename)
    for item in other_activities:
        filename = item["filename"]
        print(filename)
    
    return run_activities, other_activities

In [ ]:
def populateone_memory(df_prepop, file_df, filedate, Z3_min, Z5_min):
    """
    Populates the empty DataFrame with the data from the file
    Args:
        df_prepop (df): DataFrame to be populated
        file_df (df): DataFrame containing activity data
        filedate (str): Date of the activity
        Z3_min (int): Minimum heart rate for Z3 zone
        Z5_min (int): Minimum heart rate for Z5 zone
    Returns:
        df_postpop (df): Populated DataFrame
    """
    file_df['Distance'] = pd.to_numeric(file_df['Distance'], errors='coerce')

    df_prepop.loc[df_prepop['Date'] == filedate, 'total km'] += file_df['Distance'].iloc[-1]

    for idx, row in file_df.iloc[:-1].iterrows():
        hr = row['Avg HR']
        distance = row['Distance']
        if Z3_min <= hr < Z5_min:
            df_prepop.loc[df_prepop['Date'] == filedate, 'km Z3-4'] += distance
        elif hr >= Z5_min:
            df_prepop.loc[df_prepop['Date'] == filedate, 'km Z5-T1-T2'] += distance

    return df_prepop

In [ ]:
def populatebydate_memory(emptydf, run_activities, other_activities, Z3_min, Z5_min):
    for i in emptydf['Date']:
        for activity in run_activities:
            filedate = datetime.strptime(activity['filename'].split('_')[1], '%d-%m-%Y').strftime('%Y-%m-%d')
            if filedate == i:
                emptydf.loc[emptydf['Date'] == filedate, 'nr. sessions'] += 1
                populateone_memory(emptydf, activity['df'], filedate, Z3_min, Z5_min)

        for activity in other_activities:
            filedate = datetime.strptime(activity['filename'].split('_')[1], '%d-%m-%Y').strftime('%Y-%m-%d')
            if filedate == i:
                temp_df = activity['df']
                time_str = temp_df['Time'].iloc[-1]
                time_obj = datetime.strptime(time_str, '%H:%M:%S.%f').time()
                time_delta = timedelta(hours=time_obj.hour, minutes=time_obj.minute, seconds=time_obj.second, microseconds=time_obj.microsecond)

                hours_alternative = round(time_delta.total_seconds() / 3600, 2)
                emptydf.loc[emptydf['Date'] == filedate, 'hours alternative'] = hours_alternative

    return emptydf

In [ ]:
def initial_transform(start_date, end_date, df_memory, Z3_min = 135, Z5_min = 173):   
    """
    Main function to extract and transform data.
    """
    try:
        Z3_min = int(Z3_min)
        Z5_min = int(Z5_min)    
    except ValueError:
        print("Please enter valid numbers for heart rate zone thresholds.")

   
    empty = create_emptydf(start_date, end_date)
    r,o = read_df_memory(df_memory)
    
    df_full = populatebydate_memory(empty, r, o, Z3_min, Z5_min)
    
    # Convert to day approach format
    dfday_user = convert_to_day_approach(df_full)
    
    return dfday_user

In [ ]:
def refactor(df):
    return df

In [ ]:
def main_extract_transform_memory(start_date, end_date, df_memory, Z3_min = 135, Z5_min = 173):
    df = initial_transform(start_date, end_date, df_memory, Z3_min, Z5_min)
    refactored_df = refactor(df)


    print(refactored_df)
    return refactored_df


# Testing

In [ ]:
start_date, end_date, df_memory = main_api_call()


In [ ]:
type(start_date)

In [ ]:
start_date = str(start_date)
end_date = str(end_date)

In [ ]:
def readfiles(df_memory):
   

    run_activities = list(fpath.glob(f'*Running_*.csv'))
    all_activities = list(fpath.glob(f'*.csv'))
    set_run = set(run_activities)
    set_all = set(all_activities)
    other_activities = list(set_all-set_run)

    return run_activities,other_activities

In [22]:
empty = create_emptydf(start_date, end_date)
r,o = read_df_memory(df_memory)
    
#df_full = populatebydate_memory(empty, r, o, Z3_min, Z5_min)
    
    # Convert to day approach format
#dfday_user = convert_to_day_approach(df_full)
o

running_10-08-2025_20010223403.csv
treadmill_running_08-08-2025_19990936196.csv
running_06-08-2025_19966811412.csv
running_03-08-2025_19933730154.csv
running_01-08-2025_19918398512.csv
running_30-07-2025_19892005151.csv
running_27-07-2025_19861406160.csv
running_23-07-2025_19820531095.csv
running_20-07-2025_19789491594.csv
running_16-07-2025_19748000057.csv
running_13-07-2025_19717488337.csv
running_11-07-2025_19702623182.csv
running_11-07-2025_19702620332.csv
running_11-07-2025_19702616665.csv
running_08-07-2025_19665631140.csv
running_06-07-2025_19646710442.csv
running_05-07-2025_19637604808.csv
running_05-07-2025_19637229551.csv
running_01-07-2025_19596365344.csv
running_29-06-2025_19577257408.csv
running_27-06-2025_19563985323.csv
running_15-06-2025_19442388653.csv
walking_10-08-2025_20012275661.csv
indoor_cardio_07-08-2025_19983834305.csv
indoor_cardio_05-08-2025_19957680422.csv
indoor_cardio_04-08-2025_19948544911.csv
indoor_cardio_31-07-2025_19909024936.csv
indoor_cardio_29-07-2

[{'filename': 'walking_10-08-2025_20012275661.csv',
  'df':      Split          Time   Moving Time  Distance  Elevation Gain  Elev Loss  \
  0        1  00:23:19.149  00:14:15.000      1.00              53         53   
  1        2  00:19:42.877  00:08:39.000      0.72               2          2   
  2  Summary  00:43:02.026  00:22:54.000      1.72              55         55   
  
      Avg Pace Avg Moving Paces  Best Pace  Avg Run Cadence  Max Run Cadence  \
  0  0:23:18          0:14:15    0:04:15          20.609375            200.0   
  1  0:27:30          0:12:03    0:01:55          11.359375            139.0   
  2  0:25:03          0:13:20    0:01:55          16.375000            200.0   
  
    Avg Stride Length  Avg HR  Max HR Avg Temperature  Calories  
  0                --   104.0   131.0              --       139  
  1                --   102.0   123.0              --        92  
  2                --   103.0   131.0              --       231  },
 {'filename': 'indoor_card

In [ ]:
main_extract_transform_memory(start_date, end_date, df_memory)